#  System Dependencies & Ollama

In [1]:
import os
os.environ["DEBIAN_FRONTEND"] = "noninteractive"

# Fix any broken dpkg state
!dpkg --configure -a 2>/dev/null || true

# Install all system dependencies in one shot
!apt-get install -y -qq zstd lshw pciutils

# Install Ollama (GPU-aware this time)
!curl -fsSL https://ollama.com/install.sh | sh

# Sanity checks
!which ollama && echo "✅ Ollama ready" || echo "❌ Ollama not found"
!nvidia-smi | grep "Tesla\|A100\|L4\|T4" && echo "✅ GPU detected" || echo "⚠️ No GPU"

Selecting previously unselected package pci.ids.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../0-pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../1-libpci3_1%3a3.7.0-6_amd64.deb ...
Unpacking libpci3:amd64 (1:3.7.0-6) ...
Selecting previously unselected package lshw.
Preparing to unpack .../2-lshw_02.19.git.2021.06.19.996aaad9c7-2build1_amd64.deb ...
Unpacking lshw (02.19.git.2021.06.19.996aaad9c7-2build1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../3-pciutils_1%3a3.7.0-6_amd64.deb ...
Unpacking pciutils (1:3.7.0-6) ...
Selecting previously unselected package usb.ids.
Preparing to unpack .../4-usb.ids_2022.04.02-1_all.deb ...
Unpacking usb.ids (2022.04.02-1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../5-zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpackin

# Server, Model & Python Deps

In [2]:
import subprocess, time

# Start Ollama server in background
print("🚀 Starting Ollama server...")
subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)

# Pull model
print("📥 Downloading Qwen 3.5 9B (~5.5 GB)...")
!ollama pull qwen3.5:9b

# Install Python dependencies
print("📦 Installing Python packages...")
!pip install -q langgraph langchain-ollama langchain-core streamlit pandas plotly pyngrok

# Final verification
import subprocess
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)
print("✅ Setup Complete — T4 GPU ready")

🚀 Starting Ollama server...
📥 Downloading Qwen 3.5 9B (~5.5 GB)...

📦 Installing Python packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 79.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 98.1 MB/s eta 0:00:00ta 0:00:01
NAME          ID              SIZE      MODIFIED      
qwen3.5:9b    6488c96fa5fa    6.6 GB    8 seconds ago    

✅ Setup Complete — T4 GPU ready


## Phase 2 — Data Exploration

Before building any agent or quality tool, we must understand the raw state of each dataset. 
This phase profiles all four CSVs along four dimensions: **schema structure**, **completeness**, 
**format consistency**, and **disguised null patterns**. The anomalies discovered here directly 
motivate every deterministic tool built in Phase 3.

In [8]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spesa = pd.read_csv("data/project_data_quality/spesa.csv")
attivazioni = pd.read_csv("data/project_data_quality/attivazioniCessazioni.csv")
tipologia = pd.read_csv("data/project_anomaly_detection/TIPOLOGIA_VIAGGIATORE.csv")
allarmi = pd.read_csv("data/project_anomaly_detection/ALLARMI.csv")

datasets = {
    "spesa": spesa,
    "attivazioni": attivazioni,
    "tipologia": tipologia,
    "allarmi": allarmi
}

for name, df in datasets.items():
    print(f"[{name}]  rows={df.shape[0]}  columns={df.shape[1]}")

[spesa]  rows=7543  columns=18
[attivazioni]  rows=20102  columns=19
[tipologia]  rows=5095  columns=33
[allarmi]  rows=5080  columns=24


### 2.1 — Schema Audit: Column Naming Conventions

We inspect all column names for violations of standard naming conventions. 
A well-formed schema uses lowercase snake_case, avoids special characters, 
and never starts with a digit. Violations here will break downstream 
SQL ingestion, pandas attribute access, and API serialization.

In [1]:
import re
import pandas as pd

def audit_schema(df, name):
    issues = []
    for col in df.columns:
        flags = []
        if col != col.lower():
            flags.append("mixed_case")
        if re.search(r'[%\s-]', col):  # hyphen at end of class
            flags.append("special_char_or_space")
        if re.match(r'^\d', col):
            flags.append("starts_with_digit")
        if flags:
            issues.append({"dataset": name, "column": col, "violations": ", ".join(flags)})
    return pd.DataFrame(issues)

results = [audit_schema(df, name) for name, df in datasets.items()]
schema_issues = pd.concat(results, ignore_index=True) if results else pd.DataFrame()

if schema_issues.empty:
    print("No schema issues found.")
else:
    print(schema_issues.to_string(index=False))

NameError: name 'datasets' is not defined

# completeness profile

In [ ]:
def completeness_profile(df, name):
    DISGUISED_NULLS = {"n.d.", "nd", "n/a", "null", "none", "?", "-", "//", 
                       "unknown", "na", "missing", "", " "}
    
    result = []
    for col in df.columns:
        total = len(df)
        real_nulls = df[col].isna().sum()
        
        str_col = df[col].astype(str).str.strip().str.lower()
        disguised = str_col.isin(DISGUISED_NULLS).sum()
        
        effective_missing = real_nulls + disguised
        completeness_pct = round(100 * (1 - effective_missing / total), 2)
        
        result.append({
            "dataset": name, "column": col,
            "null_count": real_nulls,
            "disguised_null_count": disguised,
            "completeness_%": completeness_pct
        })
    return pd.DataFrame(result)

completeness = pd.concat([completeness_profile(df, name) for name, df in datasets.items()])

# Show the most problematic columns
sparse = completeness[completeness["completeness_%"] < 85].sort_values("completeness_%")
print(sparse[["dataset","column","null_count","disguised_null_count","completeness_%"]].to_string(index=False))

# format consistency detection

In [ ]:
# Focus on date/temporal columns and key categorical columns

def sample_unique_values(df, col, n=8):
    return df[col].dropna().astype(str).str.strip().unique()[:n].tolist()

# --- DATE FORMAT AUDIT ---
date_cols = {
    "spesa":       ["rata", "aggregation-time"],
    "attivazioni": ["mese", "anno", "RATA", "aggregation-time"],
    "tipologia":   ["DATA_PARTENZA", "ANNO_PARTENZA"],
    "allarmi":     ["DATA_PARTENZA", "ANNO_PARTENZA"]
}

for ds_name, cols in date_cols.items():
    df = datasets[ds_name]
    print(f"\n=== {ds_name.upper()} ===")
    for col in cols:
        if col in df.columns:
            samples = sample_unique_values(df, col)
            print(f"  [{col}]: {samples}")

# outliers and categorical anomaly scan

In [9]:
# Numerical outlier hints + impossible values
print("=== NUMERICAL ANOMALIES ===")

# spesa: spesa column with currency symbol
spesa_currency = spesa[spesa['spesa'].astype(str).str.contains('€', na=False)]
print(f"[spesa] Rows with '€' in `spesa` column: {len(spesa_currency)}")

# allarmi: negative TOT value
allarmi_neg = allarmi[pd.to_numeric(allarmi['TOT'], errors='coerce') < 0]
print(f"[allarmi] Rows with negative TOT: {len(allarmi_neg)}")

# allarmi: TOT as non-numeric string
allarmi_bad_tot = allarmi[pd.to_numeric(allarmi['TOT'], errors='coerce').isna() & allarmi['TOT'].notna()]
print(f"[allarmi] Rows with non-numeric TOT (e.g. '~1', 'N.D.', '1 voli'): {len(allarmi_bad_tot)}")
print(f"  Samples: {allarmi_bad_tot['TOT'].unique()[:6].tolist()}")

# attivazioni: extreme attivazioni/cessazioni values
att_col = pd.to_numeric(attivazioni['attivazioni'], errors='coerce')
print(f"\n[attivazioni] `attivazioni` — max={att_col.max()}, 99th pct={att_col.quantile(0.99):.0f}")
ces_col = pd.to_numeric(attivazioni['cessazioni'], errors='coerce')
print(f"[attivazioni] `cessazioni` — max={ces_col.max()}, 99th pct={ces_col.quantile(0.99):.0f}")

# tipologia: INVESTIGATI > ENTRATI (logical impossibility)
tip = tipologia.copy()
tip['ENTRATI_n']    = pd.to_numeric(tip['ENTRATI'],    errors='coerce')
tip['INVESTIGATI_n']= pd.to_numeric(tip['INVESTIGATI'],errors='coerce')
cross_issue = tip[tip['INVESTIGATI_n'] > tip['ENTRATI_n']]
print(f"\n[tipologia] Rows where INVESTIGATI > ENTRATI (logical violation): {len(cross_issue)}")

=== NUMERICAL ANOMALIES ===
[spesa] Rows with '€' in `spesa` column: 56
[allarmi] Rows with negative TOT: 7
[allarmi] Rows with non-numeric TOT (e.g. '~1', 'N.D.', '1 voli'): 156
  Samples: ['N.D.', '?', '~181', '1 voli', '~1', '~167']

[attivazioni] `attivazioni` — max=36423.0, 99th pct=2483
[attivazioni] `cessazioni` — max=28566.0, 99th pct=2640

[tipologia] Rows where INVESTIGATI > ENTRATI (logical violation): 216
